
# Arithmetic or geometric mean with a failing sensor

:class:`skbn.AnDE` can pool its components in two ways:

-   ``mean="arithmetic"`` (the standard AnDE) averages the joint
    probabilities of the components, so the component that explains an
    instance best dominates its prediction.
-   ``mean="geometric"`` averages their logarithms, so every component
    counts the same, and a few components that give a class a probability
    close to zero are enough to rule that class out.

On clean data the two rules give almost the same predictions. They
separate when part of the evidence is wrong.

The data have five ordinary attributes and one very precise sensor. In
training the sensor is always right. At test time a fraction of its readings
are faulty and point to the opposite class, while the other five attributes
still point to the true one.

The components that have the sensor as a child give the true class a
negligible probability, because the reading is many standard deviations
away from what they expect. The component that has the sensor as its
super-parent does not: it models the other attributes given the reading,
and they still fit the true class. AnDE follows that component on about
half of the faulty instances. The geometric mean averages it with the
other five and is confidently wrong on most of them, which costs it
accuracy and, above all, log loss.


In [ ]:
# Author: The scikit-bayes Developers
# SPDX-License-Identifier: BSD-3-Clause

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, log_loss

from skbn import AnDE

SENSOR_SHIFT, SENSOR_STD = 5.0, 0.8


def make_data(n_samples, rng):
    y = rng.randint(0, 2, size=n_samples)
    ordinary = rng.normal(loc=2.0 * y[:, None], scale=1.0, size=(n_samples, 5))
    sensor = rng.normal(loc=SENSOR_SHIFT * y, scale=SENSOR_STD)
    return np.column_stack([ordinary, sensor]), y


def add_faults(X, y, fraction, rng):
    """Make a fraction of the sensor readings point to the opposite class."""
    X = X.copy()
    faulty = rng.rand(len(X)) < fraction
    X[faulty, -1] = rng.normal(
        loc=SENSOR_SHIFT * (1 - y[faulty]), scale=SENSOR_STD, size=faulty.sum()
    )
    return X, faulty


rng = np.random.RandomState(42)
X_train, y_train = make_data(2000, rng)
X_test, y_test = make_data(20000, rng)

# Model, line style and marker; both are AnDE and share its colour
models = {
    "AnDE, arithmetic mean": (AnDE(n_dependence=1), "-", "o"),
    "AnDE, geometric mean": (AnDE(n_dependence=1, mean="geometric"), "--", "^"),
}
color = "#0072B2"
for model, _, _ in models.values():
    model.fit(X_train, y_train)

fractions = [0.0, 0.01, 0.02, 0.05, 0.10, 0.15]
res_ll = {name: [] for name in models}
res_acc = {name: [] for name in models}
p_true_faulty = {}

for fraction in fractions:
    X_faulty, faulty = add_faults(X_test, y_test, fraction, np.random.RandomState(0))
    for name, (model, _, _) in models.items():
        proba = model.predict_proba(X_faulty)
        res_ll[name].append(log_loss(y_test, proba))
        res_acc[name].append(accuracy_score(y_test, proba.argmax(axis=1)))
        if fraction == 0.05:
            # Probability given to the true class on the faulty readings
            p_true_faulty[name] = proba[faulty, y_test[faulty]]

for name in models:
    print(name)
    print("  Log loss:", np.round(res_ll[name], 3))
    print("  Accuracy:", np.round(res_acc[name], 3))
    print(
        "  Correct on faulty readings (5% faulty):"
        f" {np.mean(p_true_faulty[name] > 0.5):.2f}"
    )

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
percent = [100 * f for f in fractions]

for name, (_, style, marker) in models.items():
    axes[0].plot(percent, res_ll[name], color=color, ls=style, marker=marker)
    axes[1].plot(
        percent, res_acc[name], color=color, ls=style, marker=marker, label=name
    )
    axes[2].hist(
        np.log10(np.clip(p_true_faulty[name], 1e-12, 1)),
        bins=np.linspace(-12, 0, 37),
        histtype="step",
        color=color,
        ls=style,
        lw=1.8,
    )

axes[0].set_ylabel("Test log loss")
axes[1].set_ylabel("Test accuracy")
for ax in axes[:2]:
    ax.set_xlabel("Faulty sensor readings (%)")
    ax.set_xticks([0, 5, 10, 15])

axes[2].axvline(np.log10(0.5), color="#1A1A1A", ls=":", lw=1)
axes[2].set_title("Faulty readings, 5% of the test set")
axes[2].set_xlabel("Probability given to the true class")
axes[2].set_ylabel("Test instances")
axes[2].set_xticks(range(-12, 1, 2), [f"$10^{{{k}}}$" for k in range(-12, 1, 2)])

fig.legend(*axes[1].get_legend_handles_labels(), loc="outside lower center", ncol=2)
fig.suptitle("AnDE with two ways of pooling the same components")
plt.show()